# 02 · Olive groves in SIGPAC: where they are and what we measured

**SIGPAC** is Spain's agricultural parcel register: every parcel carries a land use code, and
`OV` means *olivar* (olive grove). It is the map of where olive trees are, maintained for CAP
subsidies, so it is the best free source for "which pixels are olive groves".

This notebook answers three questions with the data already published in `api/data/`
(runs anywhere, no downloads):

1. How concentrated is the olive area across municipalities? (Why 86 municipalities.)
2. How much of each municipality's SIGPAC olive area does the satellite sample actually cover?
3. What does the measured olive area look like on a map?

Two numbers from the exploration (TODO.md): SIGPAC OV totals match the Junta's yearbook within
0.7 % for Jaén (584,828 vs 588,818 ha) and 2.3 % for Granada (203,994 vs 208,835 ha).

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from oliveye.models.plots import GRID, INK, MUTED, SURFACE

DATA = Path("../api/data")
municipalities = pd.DataFrame(json.loads((DATA / "municipalities.json").read_text()))
footprints = gpd.read_file(DATA / "map.geojson")
municipalities.groupby("province_code")["olive_hectares"].agg(["count", "sum"]).round(0)

## 1. Concentration: a few municipalities hold most of the olive area

Sorting municipalities by olive area and accumulating it gives a *concentration curve*. The
pipeline processed the largest municipalities until each province reached ~80 % of its olive
area: 48 in Jaén, 38 in Granada (Granada's olive area is more spread out).

In [ ]:
PROVINCES = {"23": "Jaén", "18": "Granada"}
COLOURS = {"23": "#054d28", "18": "#2a78d6"}

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor(SURFACE)
for code, name in PROVINCES.items():
    rows = municipalities[municipalities["province_code"] == code].sort_values(
        "olive_hectares", ascending=False
    )
    share = rows["olive_hectares"].cumsum() / rows["olive_hectares"].sum() * 100
    processed = int(rows["processed"].sum())
    ax.plot(
        range(1, len(share) + 1),
        share.to_numpy(),
        color=COLOURS[code],
        linewidth=2,
        label=f"{name}: {processed} processed of {len(rows)}",
    )
    ax.scatter([processed], [share.iloc[processed - 1]], color=COLOURS[code], zorder=3)
ax.axhline(80, color=MUTED, linewidth=1, linestyle="--")
ax.set_xlabel("Municipalities, largest olive area first", color=MUTED)
ax.set_ylabel("Cumulative share of the province's olive area (%)", color=MUTED)
ax.set_ylim(0, 100)
ax.grid(color=GRID)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()

## 2. Coverage: how much of each municipality the satellite sample sees

We do not average every olive pixel: openEO aggregates **250 m cells where olive grove covers at
least 80 %** of the cell (ADR 0001 addendum). That keeps the signal pure (few roads, houses or
cereal fields mixed in) at the cost of leaving out scattered, fragmented olive groves.
Coverage = olive hectares inside those cells ÷ SIGPAC olive hectares of the municipality.

In [ ]:
keys = ["province_code", "municipality_code"]
coverage = footprints[[*keys, "name", "olive_ha", "cells"]].merge(
    municipalities[[*keys, "olive_hectares"]], on=keys
)
coverage["coverage_pct"] = (coverage["olive_ha"] / coverage["olive_hectares"] * 100).round(1)
summary = coverage.groupby("province_code").agg(
    municipalities=("name", "count"),
    cells=("cells", "sum"),
    measured_ha=("olive_ha", "sum"),
    sigpac_ha=("olive_hectares", "sum"),
    median_coverage_pct=("coverage_pct", "median"),
)
summary.rename(index=PROVINCES).round(0)

**Two different "coverages" — keep them apart.** The 86 municipalities hold ~80 % of each
province's SIGPAC olive area (municipality level). Inside them, the pure olive cells sample 79 % of
the hectares in Jaén and 59 % in Granada. Over the whole province, the satellite therefore
measures directly about 64 % of Jaén's olive area and 48 % of Granada's.

In [ ]:
# The least-covered municipalities are the fragmented ones (small parcels mixed with other uses).
coverage.sort_values("coverage_pct")[["name", "olive_hectares", "olive_ha", "coverage_pct"]].head(8)

## 3. The measured olive area on a map

Each shape is the union of a municipality's olive cells, simplified to 60 m (computed in
EPSG:25830, the metric CRS for mainland Spain, stored in EPSG:4326). This is the geometry the
web map draws.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(SURFACE)
footprints.to_crs(25830).plot(
    ax=ax,
    column="olive_ha",
    cmap="Greens",
    edgecolor=INK,
    linewidth=0.15,
    legend=True,
    legend_kwds={"label": "Measured olive area (ha)", "shrink": 0.6},
)
ax.set_axis_off()
ax.set_title("Olive area measured by Sentinel-2, 86 municipalities", loc="left", color=INK)
fig.tight_layout()

**Take-aways.** The sample covers ~80 % of each province's olive area through its largest
municipalities, and within each municipality it keeps the pure olive cells. Fragmented
municipalities (mostly in Granada) are under-sampled: a known bias, stated in the results.